# Population zonal summary

Estimate population totals within boundary polygons using a rasterised zonal-sum approach.


In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone

import geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == 'notebooks' else HERE
DATA = ROOT / 'data_raw'
OUTPUTS = ROOT / 'outputs'
TABLES = ROOT / 'tables'
for folder in (OUTPUTS, TABLES):
    folder.mkdir(parents=True, exist_ok=True)

BOUNDARY_FILE = DATA / 'boundary.gpkg'
POPULATION_RASTER = DATA / 'population.tif'
BOUNDARY_ID_FIELD = None
ALL_TOUCHED = False

if not BOUNDARY_FILE.exists():
    raise FileNotFoundError(f'Boundary file not found: {BOUNDARY_FILE}')
if not POPULATION_RASTER.exists():
    raise FileNotFoundError(f'Population raster not found: {POPULATION_RASTER}')

boundaries = gpd.read_file(BOUNDARY_FILE)
with rasterio.open(POPULATION_RASTER) as src:
    if boundaries.crs != src.crs:
        boundaries = boundaries.to_crs(src.crs)
    population = src.read(1, masked=False).astype('float64')
    if src.nodata is not None:
        population = np.where(population == src.nodata, 0.0, population)
    population = np.where(np.isfinite(population), population, 0.0)

    totals = []
    for geom in boundaries.geometry:
        mask = rasterize([(geom, 1)], out_shape=population.shape, transform=src.transform, fill=0, all_touched=ALL_TOUCHED, dtype='uint8')
        totals.append(float((population * mask).sum()))

if BOUNDARY_ID_FIELD and BOUNDARY_ID_FIELD in boundaries.columns:
    boundary_id = boundaries[BOUNDARY_ID_FIELD].astype(str)
else:
    boundary_id = [f'boundary_{i+1:03d}' for i in range(len(boundaries))]

result = pd.DataFrame({'boundary_id': boundary_id, 'population_total': totals})
result.to_csv(TABLES / 'population_by_boundary.csv', index=False)
with open(OUTPUTS / 'population_by_boundary_summary.json', 'w', encoding='utf-8') as f:
    json.dump({'run_time_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'), 'all_touched': ALL_TOUCHED, 'n_boundaries': int(len(result)), 'population_total_sum': float(result['population_total'].sum())}, f, indent=2)
result
